# IE643 - Tokenizer check + trivial baselines (CPU session)

**Accelerator:** None (CPU). **Internet:** on. **Inputs:** notebook outputs **IE643 Cleaning Step 7** (final `reports_clean.csv`) and **IE643 Cleaning** (`images_512.npy`).

1. **Tokenizer check** - do CheXagent-2 and DistilGPT2 give identical token IDs on our impressions? If yes, token-level KD (top-k logits) is possible and `CACHE_TOPK = True` is worth turning on in the teacher notebook. If not, use sequence-level + feature KD only.
2. **Baselines** on val and test: most-common training impression, random training impression, and nearest-neighbour retrieval (ImageNet ViT-Tiny features, the student's starting encoder). NLG metrics via pycocoevalcap; clinical F1 from the CheXbert labels already in the CSV (see the caveat printed with the results).

In [ ]:
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
CSV_PATH = None                       # None = use the known Step 7 output; set explicitly to override
NPY_PATH = None
EVAL_SPLITS = ["val", "test"]
RUN_TOKENIZER_CHECK = True
RUN_BASELINES = True
SEED = 0

In [ ]:
import os, sys, json, glob, re, subprocess, time
import numpy as np, pandas as pd

DEPS_SCRIPT = r"""
import ast, sys, json, glob, os, importlib.util
d, only = sys.argv[1], (sys.argv[2] if len(sys.argv) > 2 else "")
std, files = set(sys.stdlib_module_names), sorted(glob.glob(d + "/*.py"))
local = {os.path.basename(f)[:-3] for f in files}
req, opt = {}, {}
for f in files:
    if only and os.path.basename(f) not in only.split(","):
        continue
    tree = ast.parse(open(f, encoding="utf-8").read())
    def visit(node, nested):
        for ch in ast.iter_child_nodes(node):
            names = [a.name for a in ch.names] if isinstance(ch, ast.Import) else \
                    ([ch.module] if isinstance(ch, ast.ImportFrom) and ch.level == 0 and ch.module else [])
            for n in names:
                top = n.split(".")[0]
                if top not in std and top not in local:
                    (opt if nested else req).setdefault(top, []).append(os.path.basename(f))
            visit(ch, nested or isinstance(ch, (ast.Try, ast.If, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.With)))
    visit(tree, False)
def missing(mods):
    out = []
    for m in mods:
        try:
            ok = importlib.util.find_spec(m) is not None
        except Exception:
            ok = False
        if not ok:
            out.append(m)
    return sorted(out)
print(json.dumps({"missing_required": missing(req), "required": sorted(req), "optional_missing": missing(opt)}))
"""
PIP_NAME = {"cv2": "opencv-python-headless", "PIL": "pillow", "yaml": "pyyaml", "sklearn": "scikit-learn", "skimage": "scikit-image", "bs4": "beautifulsoup4"}

def resolve_deps(py, code_dir, only="", constraints=("numpy<2",)):
    """Read every .py in code_dir, find the packages it imports at module level, install what the env is missing."""
    env = dict(os.environ, UV_CACHE_DIR="/tmp/uvcache")
    for attempt in range(3):
        r = subprocess.run([py, "-c", DEPS_SCRIPT, code_dir, only], capture_output=True, text=True)
        assert r.returncode == 0, "dependency scan failed: " + r.stderr[-1500:]
        info = json.loads(r.stdout.strip().splitlines()[-1])
        if not info["missing_required"]:
            print(f"remote-code imports OK (required: {info['required']}); conditional/optional and not installed: {info['optional_missing']}")
            return info
        pk = [PIP_NAME.get(m, m) for m in info["missing_required"]]
        print("installing packages the model code imports:", pk)
        r = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py] + pk + list(constraints), capture_output=True, text=True, env=env)
        assert r.returncode == 0, "could not install " + str(pk) + ": " + r.stderr[-1500:]
    raise RuntimeError(f"still missing after installing: {info['missing_required']}")

def fetch_remote_code(py, repo, code_dir):
    """Only the small code/config files of a HF repo (no weights)."""
    r = subprocess.run([py, "-c", "import sys; from huggingface_hub import snapshot_download as s; s(sys.argv[1], allow_patterns=['*.py', '*.json', '*.txt'], local_dir=sys.argv[2])",
                        repo, code_dir], capture_output=True, text=True)
    assert r.returncode == 0, f"could not fetch code files of {repo}: " + r.stderr[-1500:]

def setup_private_env(pkgs, check_code, venv, pythons=("3.10", "3.11")):
    """Kaggle's default Python is 3.13; transformers==4.40.0 needs tokenizers 0.19 which has no 3.13 wheel.
    So CheXagent-related code runs in its own Python 3.10 environment (what the CheXagent README pins), built with uv."""
    py = f"{venv}/bin/python"
    def ok():
        return os.path.exists(py) and subprocess.run([py, "-c", check_code], capture_output=True).returncode == 0
    if ok():
        return py
    t = time.time()
    env = dict(os.environ, UV_CACHE_DIR="/tmp/uvcache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    if not os.path.exists(py):
        for v in pythons:
            r = subprocess.run([sys.executable, "-m", "uv", "venv", "--python", v, venv], capture_output=True, text=True, env=env)
            if r.returncode == 0:
                break
            print(f"uv venv python {v} failed:", r.stderr[-400:])
        else:
            raise RuntimeError("could not create a private Python environment")
    r = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py] + list(pkgs), capture_output=True, text=True, env=env)
    print(r.stdout[-1500:], r.stderr[-3000:])
    assert r.returncode == 0, "package install into the private environment failed (log above)"
    assert ok(), f"private environment check failed: {check_code}"
    print(f"private environment ready at {venv} in {time.time() - t:.0f}s")
    return py

KNOWN_CSV = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning-step-7/reports_clean.csv"   # FINAL csv (Step 7 output)
KNOWN_NPY = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning/images_512.npy"            # images live in the ORIGINAL cleaning output

def resolve_data():
    csv, npy = CSV_PATH, NPY_PATH
    if csv is None and os.path.exists(KNOWN_CSV):
        csv = KNOWN_CSV
    if npy is None and os.path.exists(KNOWN_NPY):
        npy = KNOWN_NPY
    if csv is None:                                   # fallback: any reports_clean.csv that has the Step 7 columns
        c = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True)
             if {"impression_final", "use_for_training"} <= set(pd.read_csv(p, nrows=0).columns)]
        assert len(c) == 1, f"Need exactly one reports_clean.csv containing impression_final; found {c}. Attach the 'IE643 Cleaning Step 7' notebook output."
        csv = c[0]
    if npy is None:
        c = glob.glob(f"{INPUT_ROOT}/**/images_512.npy", recursive=True)
        assert len(c) == 1, f"Need exactly one images_512.npy; found {c}. Attach the 'IE643 Cleaning' notebook output."
        npy = c[0]
    assert os.path.exists(csv) and os.path.exists(npy), f"missing {csv} or {npy}; attached inputs: {os.listdir(INPUT_ROOT)}"
    return csv, npy

CSV, NPY = resolve_data()
df = pd.read_csv(CSV, low_memory=False)
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0] and (df["idx"].values == np.arange(len(df))).all(), "CSV and images_512.npy are not aligned"
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
print(f"csv: {CSV}\nnpy: {NPY}\nrows {len(df)} | usable per split {U.subset.value_counts().to_dict()}")

LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
assert all(c in df.columns for c in LABELS), [c for c in LABELS if c not in df.columns]

## 1. Tokenizer check
Runs in a separate process with `transformers==4.40.0` installed into its own folder, so the old version CheXagent needs cannot break anything else in this kernel.

In [ ]:
TOK_SCRIPT = r'''
import sys, json
from transformers import AutoTokenizer
texts = json.load(open(sys.argv[1]))
ts = AutoTokenizer.from_pretrained("distilgpt2")
try:
    tt = AutoTokenizer.from_pretrained("StanfordAIMI/CheXagent-2-3b", trust_remote_code=True)
except Exception as e:                                   # fall back to comparing the BPE files themselves
    from huggingface_hub import hf_hub_download
    rd = lambda repo, f: open(hf_hub_download(repo, f)).read()
    va, vb = json.loads(rd("distilgpt2", "vocab.json")), json.loads(rd("StanfordAIMI/CheXagent-2-3b", "vocab.json"))
    ma, mb = rd("distilgpt2", "merges.txt").split("\n")[1:], rd("StanfordAIMI/CheXagent-2-3b", "merges.txt").split("\n")[1:]
    res = {"teacher_tokenizer_load_error": repr(e)[:600], "vocab_json_same_for_all_distilgpt2_tokens": all(vb.get(k) == v for k, v in va.items()),
           "merges_identical": [x for x in ma if x] == [x for x in mb if x], "TOKEN_LEVEL_KD_OK": None}
    print(json.dumps(res, indent=1)); json.dump(res, open(sys.argv[2], "w"), indent=1); sys.exit(0)
res = {"student_vocab": len(ts), "teacher_vocab": len(tt), "n_texts": len(texts)}
n = ts.vocab_size                                        # 50257
a = ts.convert_ids_to_tokens(list(range(n)))
b = tt.convert_ids_to_tokens(list(range(n)))
res["shared_id_range_same_token_frac"] = sum(x == y for x, y in zip(a, b)) / n
res["vocab_mismatch_examples"] = [(i, x, y) for i, (x, y) in enumerate(zip(a, b)) if x != y][:10]
for name, prep in [("plain", lambda t: t), ("leading_space", lambda t: " " + t)]:
    same, diffs, max_id = 0, [], 0
    for t in texts:
        x = ts.encode(prep(t))
        y = tt.encode(prep(t), add_special_tokens=False)
        max_id = max(max_id, max(y) if y else 0)
        if x == y:
            same += 1
        elif len(diffs) < 5:
            k = next((j for j in range(min(len(x), len(y))) if x[j] != y[j]), min(len(x), len(y)))
            diffs.append({"text": t[:150], "first_diff_pos": k, "student": ts.convert_ids_to_tokens(x[max(0, k-2):k+3]),
                          "teacher": tt.convert_ids_to_tokens(y[max(0, k-2):k+3])})
    res[name] = {"identical_frac": same / len(texts), "examples": diffs, "teacher_max_id_used": max_id}
res["eos"] = {"student": [ts.eos_token, ts.eos_token_id], "teacher": [tt.eos_token, tt.eos_token_id]}
res["TOKEN_LEVEL_KD_OK"] = bool(res["plain"]["identical_frac"] == 1.0 and res["leading_space"]["identical_frac"] == 1.0
                                and res["eos"]["student"] == res["eos"]["teacher"])
print(json.dumps(res, indent=1))
json.dump(res, open(sys.argv[2], "w"), indent=1)
'''
if RUN_TOKENIZER_CHECK:
    TOKPY = setup_private_env(["transformers==4.40.0", "sentencepiece", "protobuf", "numpy<2", "pillow", "requests", "matplotlib", "einops"],
                              "import transformers; assert transformers.__version__ == '4.40.0'", venv="/tmp/tokenv")
    fetch_remote_code(TOKPY, "StanfordAIMI/CheXagent-2-3b", "/tmp/cx_code")
    resolve_deps(TOKPY, "/tmp/cx_code", only="tokenization_chexagent.py")      # e.g. matplotlib
    open("/tmp/tok_check.py", "w").write(TOK_SCRIPT)
    json.dump(U.impression_final.tolist(), open("/tmp/tok_texts.json", "w"))
    env = dict(os.environ, TOKENIZERS_PARALLELISM="false")
    t = time.time()
    r = subprocess.run([TOKPY, "/tmp/tok_check.py", "/tmp/tok_texts.json", f"{WORK}/tokenizer_check.json"],
                       env=env, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-3000:])
    assert r.returncode == 0, "tokenizer check failed - see stderr above"
    TC = json.load(open(f"{WORK}/tokenizer_check.json"))
    if "plain" not in TC:
        print("\nVERDICT (partial): CheXagent's tokenizer class could not be loaded, so only the BPE files were compared:", TC,
              "\n-> send me this output; until then keep CACHE_TOPK = False.")
    else:
      print(f"\n({time.time() - t:.0f}s) VERDICT: token IDs identical on "
          f"{TC['plain']['identical_frac']:.2%} (plain) / {TC['leading_space']['identical_frac']:.2%} (leading space) of impressions; "
          f"same token at {TC['shared_id_range_same_token_frac']:.2%} of IDs 0-50256 ->",
          "token-level KD is possible: set CACHE_TOPK = True" if TC["TOKEN_LEVEL_KD_OK"] else "use sequence-level + feature KD; keep CACHE_TOPK = False")

## 2. Baselines

In [ ]:
import torch, torch.nn.functional as F

try:
    import pycocoevalcap
except ImportError:                                      # not part of Kaggle's default image
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)
    import pycocoevalcap
import shutil
print("pycocoevalcap ok | java:", shutil.which("java") or "NOT FOUND (METEOR will be skipped)")

def text_tokens(t):
    return " ".join(re.findall(r"[a-z0-9]+", t.lower()))   # lower-case, punctuation dropped (stated in the report)

def nlg_metrics(preds, refs):
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    g = {i: [text_tokens(r) or "empty"] for i, r in enumerate(refs)}
    p = {i: [text_tokens(x) or "empty"] for i, x in enumerate(preds)}
    out = {}
    try:
        b, _ = Bleu(4).compute_score(g, p, verbose=0)
    except TypeError:                                        # older pycocoevalcap has no verbose argument
        b, _ = Bleu(4).compute_score(g, p)
    out.update({f"BLEU-{k + 1}": float(v) for k, v in enumerate(b)})
    out["ROUGE-L"] = float(Rouge().compute_score(g, p)[0])
    out["CIDEr"] = float(Cider().compute_score(g, p)[0])
    try:
        from pycocoevalcap.meteor.meteor import Meteor
        out["METEOR"] = float(Meteor().compute_score(g, p)[0])
    except Exception as e:
        out["METEOR"] = None
        print("METEOR skipped (needs Java):", repr(e)[:200])
    return out

def label_matrix(rows):
    m = np.zeros((len(rows), len(LABELS)), dtype=np.int8)
    for j, c in enumerate(LABELS):
        if c == "No Finding" and "y_no_finding" in rows.columns:
            m[:, j] = (rows["y_no_finding"].values == 1)      # conflict-fixed version from cleaning Step 7
        else:
            m[:, j] = (rows[c].values == 1)                   # positive only; uncertain (-1) and blank count as negative
    return m

def clinical_f1(P, G, cols=LABELS):
    J = [LABELS.index(c) for c in cols]
    P, G = P[:, J].astype(bool), G[:, J].astype(bool)
    tp = (P & G).sum(0); fp = (P & ~G).sum(0); fn = (~P & G).sum(0)
    den = 2 * tp + fp + fn
    per = np.where(den > 0, 2 * tp / np.maximum(den, 1), np.nan)      # NaN = label absent in both GT and predictions
    micro = 2 * tp.sum() / max(den.sum(), 1)
    prec = tp.sum() / max(tp.sum() + fp.sum(), 1); rec = tp.sum() / max(tp.sum() + fn.sum(), 1)
    per_d = {c: (None if np.isnan(v) else float(v)) for c, v in zip(cols, per)}
    return {"micro_F1": float(micro), "micro_P": float(prec), "micro_R": float(rec), "macro_F1": float(np.nanmean(per))}, per_d

def boot_ci(P, G, cols, n=1000, seed=0):
    rs = np.random.RandomState(seed); v = []
    for _ in range(n):
        s = rs.randint(0, len(P), len(P))
        v.append(clinical_f1(P[s], G[s], cols)[0]["micro_F1"])
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

In [ ]:
def load_encoder():
    try:
        import timm
        m = timm.create_model("vit_tiny_patch16_224.augreg_in21k_ft_in1k", pretrained=True, num_classes=0)
        name = "timm vit_tiny_patch16_224.augreg_in21k_ft_in1k"
    except Exception as e:
        print("timm ViT-Tiny unavailable, falling back to torchvision ResNet-18:", repr(e)[:200])
        import torchvision
        m = torchvision.models.resnet18(weights="IMAGENET1K_V1"); m.fc = torch.nn.Identity()
        name = "torchvision resnet18 IMAGENET1K_V1"
    return m.eval(), name

def embed_cached(model, idx, name):
    p = f"{WORK}/emb_{name}.pt"
    if os.path.exists(p):
        e = torch.load(p)
        if e.shape[0] == len(idx):
            print(f"  {name}: loaded cached embeddings {tuple(e.shape)}")
            return e
    e = embed(model, idx)
    torch.save(e, p)
    return e

@torch.inference_mode()
def embed(model, idx, bs=64):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    out = []
    for s in range(0, len(idx), bs):
        x = torch.from_numpy(np.stack([np.asarray(imgs[i]) for i in idx[s:s + bs]])).float().div(255)
        if x.ndim == 3:
            x = x[:, None]
        x = F.interpolate(x, size=(224, 224), mode="bilinear", antialias=True, align_corners=False).expand(-1, 3, -1, -1)
        out.append(F.normalize(model((x - mean) / std), dim=-1))
        if (s // bs) % 20 == 0:
            print(f"  embedded {s + len(idx[s:s + bs])}/{len(idx)}", flush=True)
    return torch.cat(out)

if RUN_BASELINES:
    torch.set_num_threads(os.cpu_count())
    rs = np.random.RandomState(SEED)
    smoke = nlg_metrics(["no acute disease ."], ["no acute disease ."])      # fail now, not after the slow embedding
    print("metric smoke test:", {k: (round(v, 3) if v is not None else None) for k, v in smoke.items()})
    tr = U[U.subset == "train"].reset_index(drop=True)
    Ltr = label_matrix(tr)
    # most common training impression; its label vector = the most common label vector among rows with that text
    top_text = tr.impression_final.value_counts().index[0]
    rows_top = Ltr[tr.impression_final.values == top_text]
    vals, counts = np.unique(rows_top, axis=0, return_counts=True)
    top_labels = vals[counts.argmax()]
    print(f"most common training impression ({(tr.impression_final == top_text).mean():.1%} of train): {top_text!r}")

    enc, enc_name = load_encoder()
    t = time.time()
    print("embedding train...")
    Etr = embed_cached(enc, tr.idx.tolist(), "train")
    RESULTS = {"encoder": enc_name, "most_common_text": top_text, "splits": {},
               "notes": ["Text metrics: lower-cased, punctuation removed, pycocoevalcap corpus-level.",
                         "Clinical F1 uses the CheXbert labels shipped with CheXpert Plus for the SOURCE report of each "
                         "prediction (labelled from the full report, not only the impression). It approximates running "
                         "CheXbert on the predicted text; Week 5 evaluation runs CheXbert on all generated text.",
                         "Positive = 1; uncertain (-1) and blank count as negative. No Finding uses y_no_finding (Step 7 fixed)."]}
    for split in EVAL_SPLITS:
        ev = U[U.subset == split].reset_index(drop=True)
        if not len(ev):
            continue
        G = label_matrix(ev)
        print(f"embedding {split}...")
        Eev = embed_cached(enc, ev.idx.tolist(), split)
        nn = (Eev @ Etr.T).argmax(1).numpy()
        rnd = rs.randint(0, len(tr), len(ev))
        preds = {"most_common": ([top_text] * len(ev), np.repeat(top_labels[None], len(ev), 0)),
                 "random_train": (tr.impression_final.values[rnd].tolist(), Ltr[rnd]),
                 "nn_retrieval": (tr.impression_final.values[nn].tolist(), Ltr[nn])}
        S = {}
        for name, (txt, P) in preds.items():
            m = nlg_metrics(txt, ev.impression_final.tolist())
            c14, per14 = clinical_f1(P, G, LABELS)
            c5, _ = clinical_f1(P, G, FIVE)
            S[name] = {**m, **{f"CE14_{k}": v for k, v in c14.items()}, **{f"CE5_{k}": v for k, v in c5.items()},
                       "CE14_micro_F1_95CI": boot_ci(P, G, LABELS), "distinct_predictions": int(len(set(txt))),
                       "pct_most_common_text": float(np.mean([x == top_text for x in txt])), "per_label_F1": per14}
        RESULTS["splits"][split] = S
        pd.DataFrame({"idx": ev.idx, "gt": ev.impression_final, "most_common": preds["most_common"][0],
                      "random_train": preds["random_train"][0], "nn_retrieval": preds["nn_retrieval"][0],
                      "nn_train_idx": tr.idx.values[nn]}).to_csv(f"{WORK}/baseline_predictions_{split}.csv", index=False)
        tab = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk not in ("per_label_F1", "CE14_micro_F1_95CI")} for k, v in S.items()}).T
        print(f"\n=== {split} ({len(ev)} studies) ===")
        print(tab.round(4).to_string())
        print("\nper-label F1 (nn_retrieval):")
        print(pd.Series(S["nn_retrieval"]["per_label_F1"]).round(3).to_string())
    print(f"\nbaselines done in {(time.time() - t) / 60:.1f} min")
    json.dump(RESULTS, open(f"{WORK}/baseline_results.json", "w"), indent=1)